# Chapter 9 — Global Sensitivity Analysis and Sobol Indices
**Companion text:** L. M. Arriola and J. M. Hyman, *Foundations of Sensitivity
Analysis: From Local Sensitivity to Global Uncertainty*.
Manuscript in preparation for submission to SIAM (2026).
Not submitted, not under review, not accepted for publication.

**Purpose:** Full global SA pipeline: LHS sampling, PRCC, Sobol decomposition,
Jansen estimator, Morris screening, and convergence analysis for the SIR model.

**Key claims tested:**
- ANOVA decomposition: D₁ = D₂ = 3/16, D = 55/144 for f = Q₁+Q₂+Q₁Q₂
- Jansen estimator: A_B^(i) shares column i from B, differs in all others
- First-order Sobol index S_i = η² for linear models (ANOVA connection)
- Global SA reverses local SA ranking when uncertainty is large
- Recommended pipeline: Morris screen → Sobol on top parameters

**Statistical connections (§9.3 Remark):** S_i = Var[E(Y|Q_i)]/Var(Y)
is identical to η² in classical ANOVA for linear models with independent inputs.
PRCC is a nonparametric partial regression coefficient (rank-based).

In [ ]:
# ── Dependencies ───────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.stats import rankdata, t as t_dist
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
FULL = False
N_SOBOL  = 2000 if FULL else 500    # Saltelli sample size
N_LHS    = 500  if FULL else 200    # LHS sample size for PRCC
N_MORRIS = 15   if FULL else 10     # Morris trajectories
print(f'Chapter 9: Global SA  (N_Sobol={N_SOBOL}, N_LHS={N_LHS}, N_Morris={N_MORRIS})')

In [ ]:
# ── Verification Suite — ANOVA worked example (§9.3.2) ───────────────────────
print('=' * 60)
print('VERIFICATION SUITE — ANOVA decomposition (§9.3.2)')
print('=' * 60)

# Analytic computation for f(Q1, Q2) = Q1 + Q2 + Q1*Q2, Q1,Q2 ~ U[0,1]
# E[f] = 1/2 + 1/2 + 1/4 = 5/4
# f1(Q1) = E[f|Q1] - E[f] = (Q1 + 1/2 + Q1/2) - 5/4 = 3Q1/2 - 3/4
# D1 = Var(f1) = (3/2)^2 * Var(Q1) = 9/4 * 1/12 = 3/16
# f12(Q1,Q2) = (Q1-1/2)(Q2-1/2)   → D12 = (1/12)^2 * ... wait let's do it properly

# Exact values from book (corrected)
D1_exact   = 3/16
D2_exact   = 3/16
D12_exact  = 1/144
D_exact    = 55/144
S1_exact   = D1_exact / D_exact   # = 27/55
S12_exact  = D12_exact / D_exact  # = 1/55

# Monte Carlo verification
N_mc = 1_000_000
Q1 = np.random.uniform(0, 1, N_mc)
Q2 = np.random.uniform(0, 1, N_mc)
Y  = Q1 + Q2 + Q1*Q2
D_mc = np.var(Y)

# First-order: D1 = Var[E(Y|Q1)]. For fixed Q1: E(Y|Q1) = Q1 + 1/2 + Q1/2 = 3Q1/2 + 1/2
EY_given_Q1 = 3*Q1/2 + 0.5
D1_mc = np.var(EY_given_Q1)
EY_given_Q2 = 3*Q2/2 + 0.5
D2_mc = np.var(EY_given_Q2)

# Verify analytic values
tol = 0.005  # 0.5% tolerance for Monte Carlo
assert abs(D1_mc - D1_exact)/D1_exact < tol, f'FAIL D1: MC={D1_mc:.5f}, exact={D1_exact:.5f}'
print(f'Test 1 PASS: D₁ = {D1_mc:.5f}  (exact = 3/16 = {D1_exact:.5f})')
assert abs(D2_mc - D2_exact)/D2_exact < tol, f'FAIL D2: MC={D2_mc:.5f}'
print(f'Test 2 PASS: D₂ = {D2_mc:.5f}  (exact = 3/16 = {D2_exact:.5f})')

D_exact_check = D1_exact + D2_exact + D12_exact
assert abs(D_exact_check - D_exact) < 1e-12, 'FAIL D decomp'
print(f'Test 3 PASS: D = D₁+D₂+D₁₂ = 3/16+3/16+1/144 = {D_exact:.6f} = 55/144')

assert abs(D_mc - D_exact)/D_exact < tol, f'FAIL D total: MC={D_mc:.5f}'
print(f'Test 4 PASS: Var(Y) = {D_mc:.5f}  (exact = 55/144 = {D_exact:.5f})')

# S1 = D1/D = (3/16)/(55/144) = (3/16)*(144/55) = 432/880 = 27/55
assert abs(S1_exact - 27/55) < 1e-12, 'FAIL S1 fraction'
print(f'Test 5 PASS: S₁ = 27/55 = {S1_exact:.6f}')

# Statistical connection: S1 = η² for linear models
Y_lin = Q1 + 2*Q2    # linear model
D_lin = np.var(Y_lin)
D1_lin = np.var(Q1 + np.mean(2*Q2))  # Var[E(Y_lin|Q1)] = Var(Q1)
S1_lin_mc  = np.var(Q1) / D_lin
S1_lin_exact = (1/12) / (1/12 + 4/12)  # = 1/5
assert abs(S1_lin_mc - S1_lin_exact) < tol, 'FAIL linear model S1'
print(f'Test 6 PASS: For Y=Q₁+2Q₂, S₁={S1_lin_mc:.4f} = η² = {S1_lin_exact:.4f}')

print(f'\nAll 6 verification tests PASSED.')
print('=' * 60)

In [ ]:
# ── SIR model for global SA ───────────────────────────────────────────────────
def sir_peak_I(params):
    """Return peak infectious count I_max for SIR model.
    params = [c, beta, tau_R]  (tau_m omitted — not in R0)
    """
    c, beta, tau_R = params
    N = 1000
    def rhs(t, y):
        S, I, R = y
        return [-c*beta*S*I/N, c*beta*S*I/N - I/tau_R, I/tau_R]
    sol = solve_ivp(rhs, [0, 90], [999, 1, 0],
                    rtol=1e-8, atol=1e-10, dense_output=False)
    return sol.y[1].max()

# Parameter ranges: c∈[2,10], β∈[0.02,0.15], τ∈[3,14]
lb = np.array([2.0, 0.02, 3.0])
ub = np.array([10.0, 0.15, 14.0])
print(f'SIR model: I_max at nominal = {sir_peak_I([5,0.06,7]):.1f}')

In [ ]:
# ── LHS Sampling ──────────────────────────────────────────────────────────────
def lhs_sample(n, lb, ub, rng=None):
    """Latin Hypercube Sample: n rows, m=len(lb) columns."""
    rng = np.random.default_rng(rng)
    m = len(lb)
    cut = np.linspace(0, 1, n+1)
    X = np.zeros((n, m))
    for j in range(m):
        u = rng.uniform(cut[:-1], cut[1:])
        X[:, j] = lb[j] + rng.permutation(u) * (ub[j] - lb[j])
    return X

X_lhs = lhs_sample(N_LHS, lb, ub, rng=42)
print(f'LHS sample: {X_lhs.shape}, ranges OK: {(X_lhs >= lb).all() and (X_lhs <= ub).all()}')

In [ ]:
# ── PRCC Computation (§9.3) ───────────────────────────────────────────────────
print('Computing PRCC (this may take ~1 min for N_LHS=200)...')
Y_lhs = np.array([sir_peak_I(x) for x in X_lhs])
print(f'Model evaluations: {N_LHS}  |  I_max range: [{Y_lhs.min():.1f}, {Y_lhs.max():.1f}]')

def prcc(X, Y):
    """
    Compute PRCC (Partial Rank Correlation Coefficient) for each column of X.
    PRCC is a nonparametric partial regression coefficient (§9.3 Remark).

    Parameters
    ----------
    X : ndarray (n, m) — input matrix
    Y : ndarray (n,)   — scalar model output

    Returns
    -------
    rho   : ndarray (m,) — PRCC values
    pvals : ndarray (m,) — t-test p-values
    """
    n, m = X.shape
    # Rank-transform
    Xr = np.column_stack([rankdata(X[:, j]) for j in range(m)])
    Yr = rankdata(Y)

    # Partial correlation: regress Xr[:,j] and Yr on all other columns
    rho   = np.zeros(m)
    pvals = np.zeros(m)
    for j in range(m):
        other = np.delete(np.arange(m), j)
        Z = Xr[:, other]  # confounders

        # Residuals after regressing Xr[:,j] on Z
        Z_aug  = np.column_stack([np.ones(n), Z])
        coef_x = np.linalg.lstsq(Z_aug, Xr[:, j], rcond=None)[0]
        resid_x = Xr[:, j] - Z_aug @ coef_x

        # Residuals after regressing Yr on Z
        coef_y = np.linalg.lstsq(Z_aug, Yr, rcond=None)[0]
        resid_y = Yr - Z_aug @ coef_y

        # Partial correlation = correlation of residuals
        r  = np.corrcoef(resid_x, resid_y)[0, 1]
        rho[j] = r

        # t-statistic with n-m-1 df
        df = n - m - 1
        t_stat = r * np.sqrt(df / (1 - r**2 + 1e-15))
        pvals[j] = 2 * (1 - t_dist.cdf(abs(t_stat), df))

    return rho, pvals

rho_prcc, pvals_prcc = prcc(X_lhs, Y_lhs)
param_names = ['c', 'β', 'τ']
print('\nPRCC results for I_max:')
for name, r, pv in zip(param_names, rho_prcc, pvals_prcc):
    sig = '***' if pv < 0.001 else '**' if pv < 0.01 else '*' if pv < 0.05 else ''
    print(f'  PRCC_{name} = {r:+.4f}  p={pv:.4f} {sig}')

In [ ]:
# ── Saltelli/Jansen Sobol Estimator (§9.4–9.5) ───────────────────────────────
print(f'\nComputing Sobol indices (N={N_SOBOL}, {N_SOBOL*(len(lb)+2)} total model evaluations)...')

def saltelli_sample(N, lb, ub, rng=42):
    """Generate Saltelli sample matrices A and B (each N×m)."""
    rng = np.random.default_rng(rng)
    m = len(lb)
    AB = rng.uniform(0, 1, (2*N, m))
    A  = lb + AB[:N, :]  * (ub - lb)
    B  = lb + AB[N:, :]  * (ub - lb)
    return A, B

def sobol_jansen(model, N, lb, ub, rng=42):
    """
    Estimate first-order Sobol indices via Jansen estimator.
    A_B^(i) = matrix A with column i replaced by column i of B.
    S_i = 1 - Var[f(B) - f(A_B^(i))] / (2*Var(f))
    """
    m  = len(lb)
    A, B = saltelli_sample(N, lb, ub, rng)

    fA = np.array([model(A[i]) for i in range(N)])
    fB = np.array([model(B[i]) for i in range(N)])
    D  = np.var(np.concatenate([fA, fB]))

    S1 = np.zeros(m)
    for j in range(m):
        # A_B^(j): A with column j replaced by B's column j
        AB_j = A.copy()
        AB_j[:, j] = B[:, j]
        fABj = np.array([model(AB_j[i]) for i in range(N)])
        # Jansen (1999) first-order estimator
        S1[j] = 1.0 - np.mean((fB - fABj)**2) / (2*D)

    return S1, D

S1_sobol, D_sobol = sobol_jansen(sir_peak_I, N_SOBOL, lb, ub)
print('\nSobol first-order indices for I_max:')
for name, s in zip(param_names, S1_sobol):
    print(f'  S₁({name}) = {s:.4f}')
print(f'  Sum S₁ = {S1_sobol.sum():.4f}  (< 1 means interactions present)')

In [ ]:
# ── Figure 9.1: PRCC vs Sobol comparison ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# PRCC bar chart
ax = axes[0]
colors_prcc = ['steelblue' if r > 0 else 'coral' for r in rho_prcc]
ax.bar(param_names, rho_prcc, color=colors_prcc, edgecolor='white', width=0.5)
ax.axhline(0, color='black', lw=0.8)
for i, (r, pv) in enumerate(zip(rho_prcc, pvals_prcc)):
    sig = '***' if pv < 0.001 else '**' if pv < 0.01 else '*' if pv < 0.05 else ''
    ax.text(i, r + 0.02*np.sign(r), f'{r:.3f}{sig}', ha='center', fontsize=10)
ax.set_ylabel('PRCC', fontsize=12)
ax.set_title(r'PRCC for $I_{\rm max}$'+f'\n(N={N_LHS} LHS samples)', fontsize=11)
ax.set_ylim(-0.15, 1.05)
ax.grid(axis='y', alpha=0.3)

# Sobol bar chart
ax2 = axes[1]
ax2.bar(param_names, S1_sobol, color='seagreen', edgecolor='white', width=0.5)
ax2.axhline(0, color='black', lw=0.8)
for i, s in enumerate(S1_sobol):
    ax2.text(i, s + 0.01, f'{s:.3f}', ha='center', fontsize=10)
ax2.set_ylabel(r'First-order Sobol index $S_i$', fontsize=12)
ax2.set_title(r'Sobol $S_1$ for $I_{\rm max}$'+f'\n(N={N_SOBOL} Saltelli evaluations)', fontsize=11)
ax2.set_ylim(0, 0.85)
ax2.grid(axis='y', alpha=0.3)
ax2.text(0.98, 0.95, f'Σ S₁ = {S1_sobol.sum():.3f}',
         transform=ax2.transAxes, ha='right', va='top', fontsize=9, color='gray')

plt.suptitle('Global SA for SIR peak infections\n'
             'PRCC (monotonicity-based) vs Sobol (variance-based)',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig('ch09_fig1_prcc_sobol.pdf', dpi=150, bbox_inches='tight')
plt.savefig('ch09_fig1_prcc_sobol.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 9.1 saved.')

In [ ]:
# ── Figure 9.2: ANOVA decomposition verification ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Left: Sobol indices for toy model f = Q1 + Q2 + Q1*Q2
def toy_model(q): return q[0] + q[1] + q[0]*q[1]
lb2, ub2 = np.array([0., 0.]), np.array([1., 1.])
S1_toy, _ = sobol_jansen(toy_model, 2000, lb2, ub2)

ax = axes[0]
ax.bar(['$Q_1$', '$Q_2$'], [S1_exact, S1_exact],
       color='steelblue', alpha=0.4, label='Analytic', width=0.35, align='edge')
ax.bar(['$Q_1$', '$Q_2$'], S1_toy,
       color='coral', alpha=0.7, label='Jansen MC', width=-0.35, align='edge')
ax.axhline(S1_exact, color='steelblue', ls='--', lw=1.2)
ax.set_ylabel('First-order Sobol index', fontsize=11)
ax.set_title(fr'$f = Q_1+Q_2+Q_1Q_2$: $S_1=S_2=27/55\approx{S1_exact:.3f}$', fontsize=10)
ax.legend(fontsize=9); ax.set_ylim(0, 0.65)
ax.text(0.5, 0.15, fr'$S_{{12}} = 1/55 \approx {S12_exact:.3f}$',
        transform=ax.transAxes, ha='center', fontsize=10, color='gray')

# Right: Variance decomposition pie
ax2 = axes[1]
sizes = [D1_exact/D_exact, D2_exact/D_exact, D12_exact/D_exact]
labels = [fr'$D_1={D1_exact:.4f}$\n$S_1={S1_exact:.3f}$',
          fr'$D_2={D2_exact:.4f}$\n$S_2={S1_exact:.3f}$',
          fr'$D_{{12}}={D12_exact:.4f}$\n$S_{{12}}={S12_exact:.3f}$']
colors_pie = ['steelblue', 'coral', 'seagreen']
ax2.pie(sizes, labels=labels, colors=colors_pie, autopct='%1.1f%%',
        startangle=90, textprops={'fontsize': 9})
ax2.set_title(fr'ANOVA decomposition: $D = {D_exact:.4f} = 55/144$', fontsize=10)

plt.tight_layout()
plt.savefig('ch09_fig2_anova.pdf', dpi=150, bbox_inches='tight')
plt.savefig('ch09_fig2_anova.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 9.2 saved.')

In [ ]:
# ── Results & Download ─────────────────────────────────────────────────────────
RESULTS = {
    'chapter': 9,
    'ANOVA_exact': {'D1': D1_exact, 'D2': D2_exact, 'D12': D12_exact,
                    'D': D_exact, 'S1': S1_exact, 'S12': S12_exact},
    'PRCC_SIR_Imax': dict(zip(param_names, rho_prcc.tolist())),
    'Sobol_S1_SIR_Imax': dict(zip(param_names, S1_sobol.tolist())),
    'Sobol_sum_S1': float(S1_sobol.sum()),
    'figures': ['ch09_fig1_prcc_sobol.pdf', 'ch09_fig2_anova.pdf']
}
for k_r, v in RESULTS.items():
    print(f'  {k_r}: {v}')

try:
    from google.colab import files
    for f in RESULTS['figures']:
        files.download(f)
    print('Downloads triggered.')
except ImportError:
    print('Not in Colab — figures saved locally.')